# Assignment 09 - Customer Fraud Detection (Binary Classification)

**Dataset:** fraud_data_cleanned.csv  
**Target:** `is_fraudulent` (0 = Not Fraudulent, 1 = Fraudulent)

### Pipeline
1. Exploratory Data Analysis (EDA)
2. Data Preprocessing (Encoding + Scaling)
3. Train/Test Split
4. SMOTE (Class Balancing)
5. Model Training: Decision Tree, Random Forest, XGBoost
6. Model Comparison & Best Model Selection


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("[INFO] xgboost not installed - XGBoost model will be skipped.")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Load Data

In [ ]:
df = pd.read_csv("fraud_data_cleanned.csv")
print("Shape:", df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print()
print("--- Missing Values ---")
print(df.isnull().sum())
print()
print("--- Duplicate Rows:", df.duplicated().sum())
print()
print("--- Target Class Distribution ---")
fraud_counts = df["is_fraudulent"].value_counts()
fraud_pct = df["is_fraudulent"].value_counts(normalize=True) * 100
for val, cnt, pct in zip(fraud_counts.index, fraud_counts.values, fraud_pct.values):
    label = "Not Fraudulent" if val == 0 else "Fraudulent"
    print(f"  {val} ({label}): {cnt} samples ({pct:.2f}%)")
print()
print("--- Statistical Summary ---")
df.describe().round(2)

In [ ]:
colors = ["#2ecc71", "#e74c3c"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Exploratory Data Analysis", fontsize=16, fontweight="bold")

axes[0, 0].bar(["Not Fraud (0)", "Fraud (1)"], fraud_counts.values, color=colors, edgecolor="black")
axes[0, 0].set_title("Target Class Distribution")
axes[0, 0].set_ylabel("Count")
for i, v in enumerate(fraud_counts.values):
    axes[0, 0].text(i, v + 50, str(v), ha="center", fontweight="bold")

sns.histplot(data=df, x="age", hue="is_fraudulent", kde=True, ax=axes[0, 1],
             palette=colors, bins=25, alpha=0.6)
axes[0, 1].set_title("Age Distribution by Fraud Status")

sns.boxplot(data=df, x="is_fraudulent", y="avg_order_value", ax=axes[0, 2], palette=colors)
axes[0, 2].set_title("Avg Order Value by Fraud Status")
axes[0, 2].set_xticklabels(["Not Fraud", "Fraud"])

country_fraud = df.groupby("country")["is_fraudulent"].mean().sort_values(ascending=False)
country_fraud.plot(kind="bar", ax=axes[1, 0], color="#3498db", edgecolor="black")
axes[1, 0].set_title("Fraud Rate by Country")
axes[1, 0].set_ylabel("Fraud Rate")
axes[1, 0].tick_params(axis="x", rotation=45)

cat_fraud = df.groupby("preferred_category")["is_fraudulent"].mean().sort_values(ascending=False)
cat_fraud.plot(kind="bar", ax=axes[1, 1], color="#9b59b6", edgecolor="black")
axes[1, 1].set_title("Fraud Rate by Product Category")
axes[1, 1].set_ylabel("Fraud Rate")
axes[1, 1].tick_params(axis="x", rotation=45)

num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 2],
            linewidths=0.5, annot_kws={"size": 7})
axes[1, 2].set_title("Correlation Heatmap")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 3. Data Preprocessing

In [ ]:
X = df.drop(columns=["customer_id", "is_fraudulent"])
y = df["is_fraudulent"]

num_col = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_col = ["gender", "country", "preferred_category"]
print(f"Numerical features ({len(num_col)}):", num_col)
print(f"Categorical features ({len(cat_col)}):", cat_col)

# Scale numerical features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[num_col])
df_num = pd.DataFrame(X_num_scaled, columns=num_col, index=X.index)

# One-Hot Encode categorical features
encoder = OneHotEncoder(drop="first", sparse_output=False)
X_cat_enc = encoder.fit_transform(X[cat_col])
cat_enc_cols = encoder.get_feature_names_out(cat_col)
df_cat = pd.DataFrame(X_cat_enc, columns=cat_enc_cols, index=X.index)

# Combine
X_preprocessed = pd.concat([df_num, df_cat], axis=1)
print(f"Preprocessed feature matrix shape: {X_preprocessed.shape}")
X_preprocessed.head()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed, y, test_size=0.20, random_state=42, stratify=y)
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train:\n{y_train.value_counts()}")
print(f"y_test:\n{y_test.value_counts()}

## 5. SMOTE - Class Balancing

In [ ]:
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After  SMOTE:", pd.Series(y_train_res).value_counts().to_dict())
print(f"Resampled X_train shape: {X_train_res.shape}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
y_train.value_counts().plot(kind="bar", ax=ax[0], color=colors, edgecolor="black")
ax[0].set_title("Before SMOTE")
pd.Series(y_train_res).value_counts().plot(kind="bar", ax=ax[1], color=colors, edgecolor="black")
ax[1].set_title("After SMOTE")
plt.tight_layout()
plt.show()

## 6. Model Training & Evaluation

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
if XGB_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=100, use_label_encoder=False,
        eval_metric="logloss", random_state=42)

results = {}
for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)

    results[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1-Score": f1, "AUC-ROC": auc, "model": model, "y_pred": y_pred
    }
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"AUC-ROC  : {auc:.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred, zero_division=0)}")

## 7. Model Comparison

In [ ]:
comp_df = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k not in ("model", "y_pred")}
    for name, vals in results.items()
}).T
comp_df = comp_df.round(4)
print(comp_df.to_string())

best_name = comp_df["F1-Score"].idxmax()
print(f"\n>>> BEST MODEL (by F1-Score): {best_name} (F1={comp_df.loc[best_name, 'F1-Score']:.4f})")

# Comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("Model Comparison", fontsize=16, fontweight="bold")

comp_df[["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]].plot(
    kind="bar", ax=axes[0], colormap="viridis", edgecolor="black")
axes[0].set_title("Performance Metrics Comparison")
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(loc="lower right", fontsize=8)

for idx, (name, vals) in enumerate(results.items()):
    cm = confusion_matrix(y_test, vals["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                ax=axes[1] if idx == 0 else axes[2],
                xticklabels=["Not Fraud", "Fraud"],
                yticklabels=["Not Fraud", "Fraud"])
    ax = axes[1] if idx == 0 else axes[2]
    ax.set_title(f"{name}\nConfusion Matrix")
    ax.set_ylabel("Actual" if idx == 0 else "")

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 8. Feature Importance (Best Model)

In [ ]:
best_model = results[best_name]["model"]
if hasattr(best_model, "feature_importances_"):
    feat_imp = pd.Series(best_model.feature_importances_,
                         index=X_preprocessed.columns).sort_values(ascending=False)
    print("Top 10 Features:")
    print(feat_imp.head(10).to_string())

    plt.figure(figsize=(10, 6))
    feat_imp.head(15).plot(kind="barh", color="#2c3e50", edgecolor="black")
    plt.title(f"Top 15 Feature Importances - {best_name}", fontweight="bold")
    plt.xlabel("Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 9. Conclusion

- **Dataset:** 5000 samples, highly imbalanced (97.42% Not Fraudulent, 2.58% Fraudulent)
- **Preprocessing:** StandardScaler for numerical, OneHotEncoder for categorical features
- **SMOTE:** Applied to handle class imbalance, resampled training set from 4000 to 7794 samples
- **Models Evaluated:** Decision Tree, Random Forest, XGBoost (if available)
- **Best Model:** Identified based on F1-Score

> **Note:** Due to extreme class imbalance (only 129 fraud cases out of 5000), detection of fraud cases remains challenging. Future improvements could include: trying different SMOTE variants (BorderlineSMOTE, ADASYN), threshold tuning, ensemble methods, or cost-sensitive learning.
